# Part 4c-exact — Cournot as a true MIQP

### What the approximation cost, and where that cost stops being predictable

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/lithium-modelling/blob/main/notebooks/04c_exact_miqp.ipynb)

Part 4c piecewise-linearised the quadratic revenue to keep the model a MILP, and
argued the approximation was safe: revenue is concave, we maximise, so every
chord lies below the curve and the model can only **understate** profit.

That argument is correct and it is not the whole story. This notebook solves the
same game with revenue kept as a true quadratic and measures the difference —
first inside a single best response, then inside the equilibrium. **The two
answers have opposite signs**, and the second one is the reason this notebook
exists.

### How to read this notebook

Sections 5 to 7 are carried over from Parts 4a–4c and marked as such. Section 8
builds the exact MIQP **by hand**; it is short, because the difference from the
piecewise version is about six lines. Section 9 wraps it, section 10 validates
the approximation, section 11 is the finding, and section 14 asserts the notebook
and the `lithium` package agree to $10^{-9}$.

## 0. Setup

One cell, and it is the only place the `lithium` package appears before the final check. On Colab it
clones the repo and installs it; locally it assumes you have already run `pip install -e .` and just
moves up out of `notebooks/` so the relative data paths work.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/USERNAME/lithium-modelling.git"   # <-- edit me
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

if "google.colab" in sys.modules:
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_NAME)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"font.size": 12, "axes.grid": True, "grid.alpha": 0.3})
print(f"working directory : {Path.cwd().name}")
print(f"gurobipy          : {gp.gurobi.version()}")
print(f"pandas            : {pd.__version__}")

working directory : Advanced Opt Modeling Examples
gurobipy          : (13, 0, 2)
pandas            : 2.3.3


## 1. Licence, and what fits without one

Only **section 12** needs a full licence. Everything above it, including the
validation in sections 10 and 11 that is this notebook's actual argument, runs on
the restricted `pip` licence.

That is not luck, it is what `SMALL` is for. The free licence caps **quadratic**
models at roughly **150 variables** — the linear cap of ~2,000 is far away and
never binds here. Measured:

| configuration | periods | variables | quadratic terms | fits free licence |
|---|---|---|---|---|
| `SMALL = True`, `learning='none'` | 3 | 50 | 6 | **yes** |
| `SMALL = True`, `learning='capacity'` | 3 | 83 | 6 | **yes** |
| `SMALL = False`, `learning='none'` | 13 | 398 | 26 | no |
| `SMALL = False`, `learning='capacity'` | 13 | 541 | 26 | no |

So `SMALL = True` is the shipped default.

**Never paste a licence key into a notebook.** A key committed to a repository is
exposed the moment the repository is shared, and deleting it in a later commit
does not remove it from history — the only fix is to rotate the key. This
notebook used to carry one, and it went into this repository's first commit. The
next cell reads one from the environment or, on Colab, from the secrets panel,
and falls back to the default licence when it finds nothing.

In [2]:
import os

# A licence from the environment, or from Colab secrets. Never a literal.
# Set GRB_WLSACCESSID / GRB_WLSSECRET / GRB_LICENSEID, or add those three names
# under Colab's key icon in the left sidebar.
KEYS = ('WLSACCESSID', 'WLSSECRET', 'LICENSEID')
found = {k: os.environ.get('GRB_' + k) for k in KEYS}
if not all(found.values()):
    try:
        from google.colab import userdata
        found = {k: userdata.get('GRB_' + k) for k in KEYS}
    except Exception:
        found = {}

if found and all(found.values()):
    ENV = gp.Env(params={'WLSACCESSID': found['WLSACCESSID'],
                         'WLSSECRET': found['WLSSECRET'],
                         'LICENSEID': int(found['LICENSEID'])})
else:
    ENV = None       # gp.Model(env=None) means "use the default licence"
HAVE_WLS = ENV is not None

print("WLS environment:", "ready" if HAVE_WLS else
      "not configured - using the default licence")
print("gp.Model(env=None) means 'use the default licence', so the same code path")
print("serves both. Nothing below section 12 needs a key.")

WLS environment: not configured - using the default licence
gp.Model(env=None) means 'use the default licence', so the same code path
serves both. Nothing below section 12 needs a key.


### 1.1 The size switch

`SMALL` shrinks the horizon from 13 periods to 3. Everything else is unchanged,
so the model is the same model — just small enough to solve as a quadratic on a
restricted licence.

**It has to be set before section 3**, because the horizon is what section 3
derives every set and coefficient from.

In [3]:
SMALL = True      # False -> the full 13-period horizon; needs a full licence

print(f"SMALL = {SMALL}  ->  "
      + ("3-period horizon; every model below fits the restricted licence"
         if SMALL else
         "13-period horizon; sections 8 and 12 need a full licence"))
if not SMALL and not HAVE_WLS:
    print("\n!! SMALL is False but no WLS licence was found. The quadratic models")
    print("!! below will fail on a restricted licence. Either set SMALL = True,")
    print("!! or set GRB_WLSACCESSID / GRB_WLSSECRET / GRB_LICENSEID and re-run.")

SMALL = True  ->  3-period horizon; every model below fits the restricted licence


## 2. The instance tables

Two kinds of number go into this model and they are treated differently.

A **knob** is a scalar carrying a concept — the discount rate, the number of revenue breakpoints,
the choke price. Knobs stay written out in the cell where they are explained, because seeing
`NBP_REV = 7` next to the sentence describing what a breakpoint mesh does *is* the lesson. This
notebook hands every knob to the package explicitly in section 14, so the agreement assertion covers
them.

A **table** is instance data — many entries, indexed by the model's own sets, named nowhere in the
prose. Tables live in `data/raw/`, and both this notebook and `src/lithium/` read the same file. If
each typed its own copy, a failed assertion could not tell a typo in the data from a bug in a
constraint. Three tables, three key structures:

| file | keyed by | rows |
|---|---|---|
| `instance_base.csv` | `(stage, region)` | 6 |
| `efficiency.csv` | `stage` | 3 |
| `market.csv` | `region` | 2 |

Read the base table first and look at it as a **frame** — rows and columns — before turning it into
anything the model can index.

In [4]:
DATA = Path("data/raw")

if not (DATA / "instance_base.csv").exists():
    print("!" * 78)
    print("! data/raw/ was not found, so this notebook is FALLING BACK to generated")
    print("! numbers. Everything below will run and every figure will render, but the")
    print("! results are NOT the shipped instance and are NOT an acceptable submission.")
    print("! Fix: clone the repo (see section 0) or run this notebook from the repo root.")
    print("!" * 78)
    DATA = Path("_generated_fallback")
    DATA.mkdir(exist_ok=True)
    (DATA / "instance_base.csv").write_text(
        "stage,region,fixed,unit,opex,legacy_cap,legacy_ret\n"
        + "\n".join(f"{s},{r},1000.0,8.0,2.0,150,15"
                    for r in ("R1", "R2") for s in ("MINE", "PROC", "MFG")) + "\n")
    (DATA / "efficiency.csv").write_text(
        "stage,eta_ceil,eta_base,alpha,beta,delta_bar\n"
        "MINE,0.92,0.86,0.0,0.0,0.02\nPROC,0.95,0.80,0.03,0.01,0.05\n"
        "MFG,0.93,0.78,0.025,0.008,0.05\n")
    (DATA / "market.csv").write_text(
        "region,demand_base,demand_growth,experience0\n"
        "R1,100.0,0.008,1000.0\nR2,75.0,0.026,1000.0\n")

base = pd.read_csv(DATA / "instance_base.csv")
print(f"instance_base.csv: {len(base)} rows x {len(base.columns)} columns")
base

instance_base.csv: 6 rows x 7 columns


,stage,region,fixed,unit,opex,legacy_cap,legacy_ret
0,MINE,R1,900.0,7.0,1.20,230,11
1,PROC,R1,1500.0,11.0,2.00,205,14
2,MFG,R1,1300.0,9.5,2.40,155,18
3,MINE,R2,820.0,6.4,1.35,165,9
4,PROC,R2,1350.0,10.0,2.20,125,16
5,MFG,R2,1180.0,8.7,2.60,100,22


The other two tables. `efficiency.csv` is keyed by stage alone — a mine's yield behaviour does not
depend on which region owns it — and `market.csv` is keyed by region alone.

In [5]:
eff = pd.read_csv(DATA / "efficiency.csv")
mkt = pd.read_csv(DATA / "market.csv")
print(f"efficiency.csv: {len(eff)} rows    market.csv: {len(mkt)} rows")
display(eff)
mkt

efficiency.csv: 3 rows    market.csv: 2 rows


,stage,eta_ceil,eta_base,alpha,beta,delta_bar
0,MINE,0.92,0.86,0.000,0.000,0.02
1,PROC,0.95,0.80,0.030,0.010,0.05
2,MFG,0.93,0.78,0.025,0.008,0.05


,region,demand_base,demand_growth,experience0
0,R1,100.0,0.008,2600.0
1,R2,75.0,0.026,500.0


### 2.1 From table to lookup — see the key

A frame shows rows and columns. **The model does not index by row number.** Every constraint below
looks a value up by a *key*: `FIXED['PROC', 'R1']`, `ETA_CEIL['MFG']`, `EXPERIENCE0['R2']`. That key
is what connects the data to the algebra, so build the dictionaries and print them in the form the
constraints will actually use, rather than leaving the index set implied by punctuation.

The two index sets come out of the tables themselves, in file order — so the order the model
iterates in is a property of the data, not of a sorted() call somewhere.

In [6]:
STAGES = list(dict.fromkeys(eff["stage"]))
REGIONS = list(dict.fromkeys(mkt["region"]))

FIXED = {(r.stage, r.region): r.fixed for r in base.itertuples()}
UNIT = {(r.stage, r.region): r.unit for r in base.itertuples()}
OPEX = {(r.stage, r.region): r.opex for r in base.itertuples()}
LEGACY_CAP = {(r.stage, r.region): float(r.legacy_cap) for r in base.itertuples()}
LEGACY_RET = {(r.stage, r.region): int(r.legacy_ret) for r in base.itertuples()}

print(f"index sets:  STAGES  = {STAGES}")
print(f"             REGIONS = {REGIONS}")
print(f"{len(FIXED)} keys, each a (stage, region) tuple\n")
print(f"{'key':18s} {'FIXED':>8s} {'UNIT':>7s} {'OPEX':>6s} {'LEG_CAP':>8s} {'LEG_RET':>8s}")
for s in STAGES:
    for r in REGIONS:
        print(f"{str((s, r)):18s} {FIXED[s, r]:8.1f} {UNIT[s, r]:7.2f} {OPEX[s, r]:6.2f}"
              f" {LEGACY_CAP[s, r]:8.0f} {LEGACY_RET[s, r]:8d}")

index sets:  STAGES  = ['MINE', 'PROC', 'MFG']
             REGIONS = ['R1', 'R2']
6 keys, each a (stage, region) tuple

key                   FIXED    UNIT   OPEX  LEG_CAP  LEG_RET
('MINE', 'R1')        900.0    7.00   1.20      230       11
('MINE', 'R2')        820.0    6.40   1.35      165        9
('PROC', 'R1')       1500.0   11.00   2.00      205       14
('PROC', 'R2')       1350.0   10.00   2.20      125       16
('MFG', 'R1')        1300.0    9.50   2.40      155       18
('MFG', 'R2')        1180.0    8.70   2.60      100       22


The same move for the two single-key tables. Note `EXPERIENCE0`: R1 starts with 2,600 units of
accumulated production experience against R2's 500. Nothing in this instance is labelled
"incumbent" — that word is a *reading* of three numbers pulling the same way: R1 starts with more
experience, larger inherited plants, and the bigger home market. R2's compensating advantage is in
`instance_base.csv`, and it is capital, not operating cost: R2 is cheaper to **build** at every stage
and more expensive to **run** at every stage. Check that against the table above before going on,
because most of the results below turn on it.

In [7]:
ETA_CEIL = {r.stage: r.eta_ceil for r in eff.itertuples()}
ETA_BASE = {r.stage: r.eta_base for r in eff.itertuples()}
ALPHA = {r.stage: r.alpha for r in eff.itertuples()}
BETA = {r.stage: r.beta for r in eff.itertuples()}
DELTA_BAR = {r.stage: r.delta_bar for r in eff.itertuples()}

DEMAND_BASE = {r.region: r.demand_base for r in mkt.itertuples()}
DEMAND_GROWTH = {r.region: r.demand_growth for r in mkt.itertuples()}
EXPERIENCE0 = {r.region: r.experience0 for r in mkt.itertuples()}

print("keyed by stage :")
for s in STAGES:
    print(f"  {s!r:8s} ceil {ETA_CEIL[s]:.2f}  base {ETA_BASE[s]:.2f}  alpha {ALPHA[s]:.3f}"
          f"  beta {BETA[s]:.3f}  delta_bar {DELTA_BAR[s]:.2f}")
print("\nkeyed by region:")
for r in REGIONS:
    print(f"  {r!r:6s} demand_base {DEMAND_BASE[r]:6.1f}  growth {DEMAND_GROWTH[r]:.3f}"
          f"  experience0 {EXPERIENCE0[r]:7.1f}")

keyed by stage :
  'MINE'   ceil 0.92  base 0.86  alpha 0.000  beta 0.000  delta_bar 0.02
  'PROC'   ceil 0.95  base 0.80  alpha 0.030  beta 0.010  delta_bar 0.05
  'MFG'    ceil 0.93  base 0.78  alpha 0.025  beta 0.008  delta_bar 0.05

keyed by region:
  'R1'   demand_base  100.0  growth 0.008  experience0  2600.0
  'R2'   demand_base   75.0  growth 0.026  experience0   500.0


### 2.2 Want to experiment? Change a number here, not in the CSV

Each table is now a dictionary keyed the way the model indexes. Assign to a key to override one
entry. The change flows into every model built below **and** into the section 14 agreement check,
because the package takes the data as an argument instead of re-reading the file — so the assertion
stays green and you can see exactly what your edit did.

In [8]:
# ---------------------------------------------------------------------------
# Example - give R2 the same processing opex as R1 (2.0 instead of its 2.2):
#
#     OPEX['PROC', 'R2'] = 2.00
#
# Uncomment, then re-run this cell and everything below it. This erases R1's
# operating-cost edge at the processing stage, so R2 gets cheaper to run, its
# equilibrium share rises, and the section 14 assertion stays green because the
# notebook passes OPEX to the package rather than letting it re-read the file.
# Re-run with it commented out to get the shipped instance back.
# ---------------------------------------------------------------------------
print(f"OPEX['PROC', 'R2'] is currently {OPEX['PROC', 'R2']}")

OPEX['PROC', 'R2'] is currently 2.2


## 3. The knobs, and the structure derived from them

Everything from here to section 5 is **derived**: sets, windows, discount weights, yields. None of
it is data and none of it is a knob — it is arithmetic on the two, and doing that arithmetic is the
point of this section. `src/lithium/structure.py` computes exactly the same things, and section 14
is what proves the two derivations agree.

### 3.1 Time: variable-length periods

**`BLOCKS` follows `SMALL` here**, which is the one place this notebook differs
structurally from Part 4c. At `SMALL = True` the horizon is 3 periods over 5
years, small enough for the exact MIQP to fit a restricted licence; at `False` it
is the full 13 periods over 37 years that every other Part 4 notebook uses. Nothing
else changes, so the model is the same model at either size. `OMEGA[p]` is the sum of discount factors for the
years inside period `p`, so a long late period is correctly worth less per year *and* covers more
years. The cell prints the block structure it actually built, so you can check it against whatever
`BLOCKS` says rather than against this paragraph.

In [9]:
BLOCKS = [(2, 1), (1, 3)] if SMALL else [(6, 1), (4, 3), (2, 5), (1, 9)]   # (how many periods, how many years each)
DR = 0.05                                    # discount rate

LEN, START = [], []
_y = 1
for _count, _length in BLOCKS:
    for _ in range(_count):
        LEN.append(_length)
        START.append(_y)
        _y += _length

P = list(range(len(LEN)))
HORIZON = _y - 1
YEARS = {p: list(range(START[p], START[p] + LEN[p])) for p in P}
OMEGA = {p: sum(1 / (1 + DR) ** t for t in YEARS[p]) for p in P}
YEAR_TO_P = {t: p for p in P for t in YEARS[p]}

print(f"{len(P)} periods covering {HORIZON} years")
print(f"{'p':>3s} {'start':>6s} {'len':>4s} {'OMEGA':>8s}")
for p in P:
    print(f"{p:3d} {START[p]:6d} {LEN[p]:4d} {OMEGA[p]:8.4f}")

3 periods covering 5 years
  p  start  len    OMEGA
  0      1    1   0.9524
  1      2    1   0.9070
  2      3    3   2.4701


### 3.2 Technology: turning a lump of capex into an annual charge

A plant built in period `v` is paid for once and used for `LIFE` years. `CRF` is the capital
recovery factor that spreads that lump into equal annual payments, and `MU[s, v]` discounts those
payments back to today — truncated at the horizon, so a plant built late gets credit only for the
years that actually fall inside the model.

`LEAD` is the construction lag: decide in period `v`, produce from `START[v] + LEAD[s]`.

In [10]:
LIFE = 25
LEAD = {'MINE': 1, 'PROC': 2, 'MFG': 2}
CAP_MIN, CAP_MAX = 60.0, 260.0

CRF = DR * (1 + DR) ** LIFE / ((1 + DR) ** LIFE - 1)
ONLINE = {(s, p): START[p] + LEAD[s] for s in STAGES for p in P}
MU = {(s, v): CRF * sum(1 / (1 + DR) ** t
                        for t in range(ONLINE[s, v], ONLINE[s, v] + LIFE)
                        if t <= HORIZON)
      for s in STAGES for v in P}

print(f"CRF = {CRF:.6f}   (a 1.0 lump becomes {CRF:.4f} per year for {LIFE} years)")
print(f"\nMU['PROC', v] by build period - note the collapse as v approaches the horizon:")
print("  " + "  ".join(f"v{v}:{MU['PROC', v]:5.2f}" for v in P))

CRF = 0.070952   (a 1.0 lump becomes 0.0710 per year for 25 years)

MU['PROC', v] by build period - note the collapse as v approaches the horizon:
  v0: 0.18  v1: 0.11  v2: 0.06


### 3.3 Efficiency: yield depends on when a plant was built and how old it is

Two effects, and they pull in opposite directions. A plant built later starts closer to the
technological ceiling (`ALPHA` — the frontier improves). A plant that has been running for a while
also drifts up towards the ceiling through operating experience (`BETA`), but only by at most
`DELTA_BAR` above where it started. `ETA_FLOOR` stops the arithmetic producing a nonsense yield.

`LEGACY_BYR = -8` says the inherited plants were built eight years before year 1 — which is why they
sit well below the frontier.

In [11]:
ETA_FLOOR = 0.60
LEGACY_BYR = -8

VINTAGES = [-1] + P                      # -1 is the inherited fleet
BYEAR = {v: (LEGACY_BYR if v == -1 else START[v]) for v in VINTAGES}

ETA = {}
for s in STAGES:
    for v in VINTAGES:
        fr = ETA_CEIL[s] - (ETA_CEIL[s] - ETA_BASE[s]) * (1 - ALPHA[s]) ** (BYEAR[v] - 1)
        fr = max(ETA_FLOOR, min(fr, ETA_CEIL[s]))
        for p in P:
            age = max(0, START[p] - BYEAR[v])
            aged = ETA_CEIL[s] - (ETA_CEIL[s] - fr) * (1 - BETA[s]) ** age
            ETA[s, v, p] = max(ETA_FLOOR, min(fr + DELTA_BAR[s], aged))

print(f"{len(ETA)} yields, keyed (stage, vintage, period)\n")
print("PROC yield in period 0, by vintage - the frontier effect:")
print("  legacy (v=-1): %.4f" % ETA['PROC', -1, 0])
print("  " + "  ".join(f"v{v}:{ETA['PROC', v, 0]:.4f}" for v in P[:5]))
print("\nlegacy PROC plant ageing through the horizon:")
print("  " + "  ".join(f"p{p}:{ETA['PROC', -1, p]:.4f}" for p in P[:6]))

36 yields, keyed (stage, vintage, period)

PROC yield in period 0, by vintage - the frontier effect:
  legacy (v=-1): 0.7698
  v0:0.8000  v1:0.8045  v2:0.8089

legacy PROC plant ageing through the horizon:
  p0:0.7698  p1:0.7716  p2:0.7733


### 3.4 Demand

Each region's demand grows from its own base at its own rate. R2 is the smaller market but grows
more than three times as fast, which is why the entrant's home turf is worth fighting for later even
though it is worth less now. The value stored is the **average annual** demand across the period, so
it is comparable across periods of different length.

In [12]:
DEMAND = {(r, p): sum(DEMAND_BASE[r] * (1 + DEMAND_GROWTH[r]) ** (t - 1) for t in YEARS[p]) / LEN[p]
          for r in REGIONS for p in P}

print(f"average annual demand ({len(DEMAND)} keys, (region, period)):")
print(f"{'p':>3s} {'year':>5s} " + " ".join(f"{r:>8s}" for r in REGIONS))
for p in P:
    print(f"{p:3d} {START[p]:5d} " + " ".join(f"{DEMAND[r, p]:8.2f}" for r in REGIONS))

average annual demand (6 keys, (region, period)):
  p  year       R1       R2
  0     1   100.00    75.00
  1     2   100.80    76.95
  2     3   102.42    81.02


### 3.5 Who can produce, when

Three sets that every constraint below indexes over, and they are worth printing rather than
trusting:

- `ACTIVE[r]` — the `(stage, vintage, period)` triples that exist at all: an inherited plant until
  its retirement year, a new plant from the period it comes online until it has run for `LIFE` years.
- `VIN[r, s, p]` — the vintages available at one stage in one period. This is the set the chain
  balance sums over.
- `BUILD[r]` — the `(stage, vintage)` pairs that could be built, i.e. those that come online before
  the horizon ends.

In [13]:
ACTIVE = {r: [(s, v, p) for s in STAGES for v in VINTAGES for p in P
              if (v == -1 and START[p] <= LEGACY_RET[s, r])
              or (v >= 0 and ONLINE[s, v] <= START[p] <= ONLINE[s, v] + LIFE - 1)]
          for r in REGIONS}
VIN = {(r, s, p): [v for (ss, v, pp) in ACTIVE[r] if (ss, pp) == (s, p)]
       for r in REGIONS for s in STAGES for p in P}
BUILD = {r: [(s, v) for s in STAGES for v in P if ONLINE[s, v] <= HORIZON]
         for r in REGIONS}

for r in REGIONS:
    print(f"{r}: {len(ACTIVE[r]):4d} active (stage, vintage, period) triples, "
          f"{len(BUILD[r]):3d} buildable (stage, vintage) pairs")
print(f"\nVIN['R1', 'MFG', p] - vintages that can manufacture, period by period:")
for p in P:
    print(f"  p{p:<2d} {VIN['R1', 'MFG', p]}")

R1:   14 active (stage, vintage, period) triples,   9 buildable (stage, vintage) pairs
R2:   14 active (stage, vintage, period) triples,   9 buildable (stage, vintage) pairs

VIN['R1', 'MFG', p] - vintages that can manufacture, period by period:
  p0  [-1]
  p1  [-1]
  p2  [-1, 0]


### 3.6 The remaining knobs: moving goods and the two penalties

`TRANSPORT` is nearly five times more expensive across regions than within one. That single number
is what makes geography matter — without it the two markets would collapse into one.

In [14]:
TRANSPORT_OWN, TRANSPORT_CROSS = 0.5, 2.4
TRANSPORT = {(rf, rt): (TRANSPORT_OWN if rf == rt else TRANSPORT_CROSS)
             for rf in REGIONS for rt in REGIONS}

PRICE_FIXED = 12.0     # the Part 4b price; kept so 4b's revenue term stays available
PEN_SHORT = 90.0       # planner-only: cost of leaving demand unserved
PEN_DISPOSE = 12.0     # cost of destroying output rather than selling it

print("TRANSPORT, keyed (from, to):")
for k, v in TRANSPORT.items():
    print(f"  {str(k):14s} {v:4.1f}")
print(f"\nPEN_DISPOSE = {PEN_DISPOSE}  <- watch this one: section 10 shows it never binds")

TRANSPORT, keyed (from, to):
  ('R1', 'R1')    0.5
  ('R1', 'R2')    2.4
  ('R2', 'R1')    2.4
  ('R2', 'R2')    0.5

PEN_DISPOSE = 12.0  <- watch this one: section 10 shows it never binds


## 4. The capacity-learning curve

Building capacity gets cheaper the more of it you have built. A learning rate `LR_CAPEX = 0.15`
means unit cost falls 15% per doubling of cumulative capacity, which is the exponent
$-\log_2(1 - 0.15)$.

**The model needs the area under that curve, not the curve itself.** Total spend to go from
`Q_START` to `Q` is the integral of the unit cost over that range — building the 401st unit costs
what the curve says at 401, not what it said at 300. So we integrate numerically (trapezoid rule,
`PANELS` panels) at each of `NBP` breakpoints, and the model interpolates between them.

**This curve is concave and it enters a cost we are *minimising*** — so a chord between two
breakpoints lies *below* the true cumulative cost, and a free convex combination would happily mix
distant breakpoints to claim a discount that does not exist. That is why the model in section 5 adds
**SOS2** here. Compare section 7, where the same concave shape in a *maximisation* needs nothing.

In [15]:
import math

LEARN_STAGES = ['PROC', 'MFG']
LR_CAPEX = 0.15        # cost falls 15% per doubling
Q_START = 300.0        # cumulative capacity at which learning starts
Q_ADD = 700.0          # how much further the mesh reaches
CAPEX_FLOOR = 0.60     # the multiplier cannot fall below this
NBP = 9                # breakpoints on the capex mesh
PANELS = 400           # trapezoid panels per breakpoint

_bc = -math.log2(1 - LR_CAPEX)
K = list(range(NBP))                     # the breakpoint index the model sums over
QBP = [Q_START + Q_ADD * k / (NBP - 1) for k in K]

CBP = []
for q in QBP:
    if q <= Q_START:
        CBP.append(0.0)
        continue
    h = (q - Q_START) / PANELS
    grid = [Q_START + i * h for i in range(PANELS + 1)]
    unit = [max(CAPEX_FLOOR, (g / Q_START) ** (-_bc)) for g in grid]
    CBP.append(sum(0.5 * (unit[i] + unit[i + 1]) * h for i in range(PANELS)))

print(f"learning exponent b = {_bc:.4f}\n")
print(f"{'k':>2s} {'QBP (cum capacity)':>19s} {'unit mult':>10s} {'CBP (cum spend mult)':>21s}")
for k in range(NBP):
    print(f"{k:2d} {QBP[k]:19.1f} {max(CAPEX_FLOOR, (QBP[k] / Q_START) ** (-_bc)):10.4f}"
          f" {CBP[k]:21.2f}")

learning exponent b = 0.2345

 k  QBP (cum capacity)  unit mult  CBP (cum spend mult)
 0               300.0     1.0000                  0.00
 1               387.5     0.9418                 84.82
 2               475.0     0.8979                165.22
 3               562.5     0.8630                242.20
 4               650.0     0.8342                316.42
 5               737.5     0.8099                388.32
 6               825.0     0.7888                458.24
 7               912.5     0.7704                526.44
 8              1000.0     0.7541                593.12


## 5. The chain, built by hand

A **region** here is a whole vertically-integrated business: it mines ore, processes it, manufactures
the finished good, and sells into either market. The next seven cells build that chain for both
regions, one block of constraints at a time.

We build it first as a **cooperative planner** — one decision maker minimising the weighted sum of
both regions' costs subject to meeting demand. That is Part 4a's model, and we need it here for a
specific reason given in section 6: it tells us the *scale* at which this chain operates, and the
operating-cost learning tiers have to be calibrated against a scale.

### 5.1 The decision variables

Seven families per region. `b` is the only binary block — build or don't — and everything else is
continuous, which is a deliberate choice from Part 3 that pays off in Part 4d.

In [16]:
m = gp.Model("planner")
m.Params.OutputFlag = 0
m.Params.MIPGap = 0.005

b, c, x, f_mp, f_pf, sale, disp = {}, {}, {}, {}, {}, {}, {}
for r in REGIONS:
    b[r] = m.addVars(BUILD[r], vtype=GRB.BINARY, name=f'b_{r}')      # build it?
    c[r] = m.addVars(BUILD[r], lb=0.0, ub=CAP_MAX, name=f'c_{r}')    # how big?
    x[r] = m.addVars(ACTIVE[r], lb=0.0, name=f'x_{r}')               # throughput
    f_mp[r] = m.addVars(P, lb=0.0, name=f'fmp_{r}')                  # mine -> proc flow
    f_pf[r] = m.addVars(P, lb=0.0, name=f'fpf_{r}')                  # proc -> mfg flow
    sale[r] = m.addVars(REGIONS, P, lb=0.0, name=f'sale_{r}')        # sales into each market
    disp[r] = m.addVars(P, lb=0.0, name=f'disp_{r}')                 # destroyed output

m.update()
print(f"{m.NumVars} variables, of which {m.NumBinVars} binary")
print(f"per region: {len(BUILD['R1'])} build pairs, {len(ACTIVE['R1'])} throughput triples")

Set parameter Username


Set parameter LicenseID to value 2750151


Academic license - for non-commercial use only - expires 2026-12-04


94 variables, of which 18 binary
per region: 9 build pairs, 14 throughput triples


### 5.2 Capacity: a plant you did not build cannot run

Three constraints, and the first two are the classic big-M pair that ties a continuous size to a
binary decision. `c <= CAP_MAX * b` forces size to zero when you don't build; `c >= CAP_MIN * b`
forces a *minimum viable scale* when you do — you cannot build a token 3-unit plant. Together they
make capacity a semi-continuous variable.

The third says throughput never exceeds the capacity available: inherited plants are capped by their
legacy size, new ones by whatever `c` was chosen.

In [17]:
for r in REGIONS:
    m.addConstrs((c[r][s, v] <= CAP_MAX * b[r][s, v] for (s, v) in BUILD[r]), name=f'su_{r}')
    m.addConstrs((c[r][s, v] >= CAP_MIN * b[r][s, v] for (s, v) in BUILD[r]), name=f'sl_{r}')
    m.addConstrs((x[r][s, v, p] <= (LEGACY_CAP[s, r] if v == -1 else c[r][s, v])
                  for (s, v, p) in ACTIVE[r]), name=f'cap_{r}')

m.update()
print(f"{m.NumConstrs} constraints after the capacity block")

64 constraints after the capacity block


### 5.3 The chain balance: what comes out of one stage goes into the next

Five constraints per region per period, and they are the physical heart of the model. Ore mined,
multiplied by the mine's **yield**, has to equal the flow into processing; that flow has to equal
what processing takes in; and so on down to manufactured output, which must equal what is sold plus
what is thrown away.

The yield `ETA[s, v, p]` is where section 3.3's work shows up: an old inherited plant loses more of
the material at every step, so it needs more ore to deliver the same finished tonne.

In [18]:
for r in REGIONS:
    m.addConstrs((gp.quicksum(ETA['MINE', v, p] * x[r]['MINE', v, p]
                              for v in VIN[r, 'MINE', p]) == f_mp[r][p] for p in P),
                 name=f'mine_{r}')
    m.addConstrs((f_mp[r][p] == gp.quicksum(x[r]['PROC', v, p] for v in VIN[r, 'PROC', p])
                  for p in P), name=f'pin_{r}')
    m.addConstrs((gp.quicksum(ETA['PROC', v, p] * x[r]['PROC', v, p]
                              for v in VIN[r, 'PROC', p]) == f_pf[r][p] for p in P),
                 name=f'pout_{r}')
    m.addConstrs((f_pf[r][p] == gp.quicksum(x[r]['MFG', v, p] for v in VIN[r, 'MFG', p])
                  for p in P), name=f'min_{r}')
    m.addConstrs((gp.quicksum(ETA['MFG', v, p] * x[r]['MFG', v, p]
                              for v in VIN[r, 'MFG', p])
                  == sale[r].sum('*', p) + disp[r][p] for p in P), name=f'mout_{r}')

m.update()
print(f"{m.NumConstrs} constraints after the chain balance")

94 constraints after the chain balance


### 5.4 Cumulative production, and the head start

`cum[p]` is everything the region has ever manufactured up to and including period `p`, undiscounted
— learning does not care about the time value of money — **plus `EXPERIENCE0[r]`**, the experience it
walked in with. Note `LEN[q] * x[...]`: a period of length 5 produces five years' worth.

This single variable is the one that turns quantity into a strategic weapon later. Selling more today
moves you up this curve, and section 10 is where that stops being a footnote.

In [19]:
cum = {}
for r in REGIONS:
    cum[r] = m.addVars(P, lb=0.0, ub=3 * CAP_MAX * HORIZON + EXPERIENCE0[r], name=f'cum_{r}')
    m.addConstrs((cum[r][p] == EXPERIENCE0[r] +
                  gp.quicksum(LEN[q] * x[r]['MFG', v, q] for q in P if q <= p
                              for v in VIN[r, 'MFG', q]) for p in P), name=f'cp_{r}')

m.update()
print("starting experience:", {r: EXPERIENCE0[r] for r in REGIONS})
print(f"upper bound on cum: ", {r: round(3 * CAP_MAX * HORIZON + EXPERIENCE0[r], 0) for r in REGIONS})

starting experience: {'R1': 2600.0, 'R2': 500.0}
upper bound on cum:  {'R1': 6500.0, 'R2': 4400.0}


### 5.5 Capital cost, and the SOS2 that section 4 warned about

Capex splits in two. Stages that do **not** learn are charged the ordinary way: an annuitised fixed
cost for deciding to build, plus an annuitised per-unit cost for the size.

Stages that **do** learn get the piecewise curve. `Q[p]` is cumulative learning-stage capacity by
period `p`, `Cc[p]` the cumulative spend multiplier read off the curve at that point, and `lam` the
convex-combination weights. The capex charged in period `p` is the *increment* `Cc[p] - Cc[p-1]` —
what this period's building added, not the whole area again.

**`m.addSOS(GRB.SOS_TYPE2, ...)` is the line that matters.** Without it the weights are free to
combine breakpoint 0 with breakpoint 8 and report a cost below the true curve, because the curve is
concave and this term is being minimised. SOS2 restricts the weights to at most two *adjacent*
breakpoints, which is exactly the interpolation we meant.

In [20]:
capex = {}
for r in REGIONS:
    capex[r] = (gp.quicksum(MU[s, v] * FIXED[s, r] * b[r][s, v] for (s, v) in BUILD[r])
                + gp.quicksum(MU[s, v] * UNIT[s, r] * c[r][s, v]
                              for (s, v) in BUILD[r] if s not in LEARN_STAGES))

    Q = m.addVars(P, lb=Q_START, ub=Q_START + Q_ADD, name=f'Q_{r}')
    Cc = m.addVars(P, lb=0.0, name=f'C_{r}')
    lam = m.addVars(P, K, lb=0.0, ub=1.0, name=f'lam_{r}')
    m.addConstrs((lam.sum(p, '*') == 1 for p in P), name=f'sc_{r}')
    m.addConstrs((Q[p] == gp.quicksum(QBP[k] * lam[p, k] for k in K) for p in P), name=f'sQ_{r}')
    m.addConstrs((Cc[p] == gp.quicksum(CBP[k] * lam[p, k] for k in K) for p in P), name=f'sC_{r}')
    m.addConstrs((Q[p] == Q_START + gp.quicksum(c[r][s, v] for (s, v) in BUILD[r]
                                                if s in LEARN_STAGES and v <= p)
                  for p in P), name=f'cc_{r}')
    for p in P:
        m.addSOS(GRB.SOS_TYPE2, [lam[p, k] for k in K])          # <-- the important line

    rate = sum(UNIT[s, r] for s in LEARN_STAGES) / len(LEARN_STAGES)
    capex[r] += gp.quicksum(MU['PROC', p] * rate * (Cc[p] - (Cc[p - 1] if p > 0 else 0.0))
                            for p in P)

m.update()
print(f"{m.NumVars} variables, {m.NumConstrs} constraints, {m.NumSOS} SOS2 sets")
print(f"({len(P)} periods x {len(REGIONS)} regions = {len(P) * len(REGIONS)} SOS2 sets, as expected)")

166 variables, 124 constraints, 6 SOS2 sets
(3 periods x 2 regions = 6 SOS2 sets, as expected)


### 5.6 Operating cost, transport and disposal

Here the operating cost is **flat** — `OPEX[s, r]` per unit, no tiers. That is deliberate and
temporary: the tiered version needs a calibration this model has not produced yet. Section 6 does the
calibration and section 8 builds the tiered version.

`OMEGA[p]` appears on every operating term because these are annual flows inside a multi-year period,
where the capex terms above used `MU` because they are annuities on a lump.

In [21]:
cost = {}
for r in REGIONS:
    opex = gp.quicksum(OMEGA[p] * OPEX[s, r] * x[r][s, v, p] for (s, v, p) in ACTIVE[r])
    trans = gp.quicksum(OMEGA[p] * TRANSPORT[r, rt] * sale[r][rt, p]
                        for rt in REGIONS for p in P)
    dcost = gp.quicksum(OMEGA[p] * PEN_DISPOSE * disp[r][p] for p in P)
    cost[r] = capex[r] + opex + trans + dcost

print("cost expression built for", list(cost))
print(f"each has {cost['R1'].size()} linear terms")

cost expression built for ['R1', 'R2']
each has 40 linear terms


### 5.7 Meet demand, then solve

The planner has to serve both markets or pay `PEN_SHORT` per unit short. `w1 = 0.5` weights the two
regions equally.

> **Predict before you run.** R1 has the bigger market and far more accumulated experience; R2 is
> cheaper to build and cheaper to run at every stage. Write down which region you expect the planner
> to lean on for the *marginal* tonne, and whether you expect any demand to go unserved at a penalty
> of 90 against a transport cost of 2.4.

The `assert` before `optimize()` is a shape check, not a status check: an empty model also "succeeds".

In [22]:
W1 = 0.5
short = m.addVars(REGIONS, P, lb=0.0, name='short')
m.addConstrs((gp.quicksum(sale[r][rt, p] for r in REGIONS) + short[rt, p] >= DEMAND[rt, p]
              for rt in REGIONS for p in P), name='demand')
pen = gp.quicksum(OMEGA[p] * PEN_SHORT * short[rt, p] for rt in REGIONS for p in P)
m.setObjective(W1 * cost['R1'] + (1 - W1) * cost['R2'] + pen, GRB.MINIMIZE)

m.update()
assert m.NumVars > 0 and m.NumConstrs > 0, "empty model"
assert m.NumSOS == len(P) * len(REGIONS), "SOS2 sets went missing"

m.optimize()
assert m.SolCount > 0, f"no solution; status {m.Status}"
print(f"status {m.Status}, objective {m.ObjVal:,.1f}, MIP gap {m.MIPGap:.2e}")
print(f"unserved demand total: {sum(short[rt, p].X for rt in REGIONS for p in P):.3f}")
print("builds:", {r: sum(1 for k in BUILD[r] if b[r][k].X > 0.5) for r in REGIONS})

status 2, objective 3,870.1, MIP gap 0.00e+00
unserved demand total: 0.000
builds: {'R1': 0, 'R2': 0}


## 6. Calibrating the operating-cost tiers

Operating cost falls with cumulative production too, but through a different mechanism: a **step
function**, not a smooth curve. Below a threshold you pay full price; past it you drop to a cheaper
tier. `LR_OPEX = 0.18` sets the size of each step (18% cheaper per tier), `N_TIERS = 3` how many
there are.

The thresholds cannot be knobs, because a threshold is only meaningful relative to how much this
chain actually produces. So they are calibrated: take the cumulative production the planner just
reached, and place the first threshold at one eighth of it, the second at a quarter. Any region that
runs its chain hard passes both; one that idles passes neither.

**`LAG_YEARS = 3`** is the other idea here. Learning takes time to show up in the cost base, so the
tier applying in period `p` is decided by cumulative production three years *earlier*. That lag is
what stops a firm buying itself an instant discount in the period it produces.

In [23]:
LR_OPEX = 0.18        # each tier is 18% cheaper than the one before
OPEX_FLOOR = 0.65     # the multiplier cannot fall below this
N_TIERS = 3
LAG_YEARS = 3

top = {r: cum[r][P[-1]].X for r in REGIONS}

TIER_Q, TIER_M = {}, {}
for r in REGIONS:
    _t = max(top[r], 1.0)
    _q1 = _t / 8.0
    TIER_Q[r] = [_q1 * 2 ** j for j in range(N_TIERS - 1)]
    TIER_M[r] = [max(OPEX_FLOOR, (1 - LR_OPEX) ** j) for j in range(N_TIERS)]

print("cumulative production the planner reached, by the last period:")
for r in REGIONS:
    print(f"  {r}: {top[r]:10.2f}")
print(f"\n{N_TIERS} tiers means {N_TIERS - 1} thresholds (they are the boundaries between tiers):")
for r in REGIONS:
    print(f"  {r}: thresholds {[round(q, 1) for q in TIER_Q[r]]}"
          f"   multipliers {[round(v, 3) for v in TIER_M[r]]}")

cumulative production the planner reached, by the last period:
  R1:    3310.73
  R2:     982.67

3 tiers means 2 thresholds (they are the boundaries between tiers):
  R1: thresholds [413.8, 827.7]   multipliers [1.0, 0.82, 0.672]
  R2: thresholds [122.8, 245.7]   multipliers [1.0, 0.82, 0.672]


## 7. Carried over from Part 4c: inverse demand and the piecewise mesh

Two things this notebook needs in order to have something to compare against.
Both are narrated in `04c_cournot.ipynb` sections 7.1 and 7.5; nothing here is
new, and section 8 is where this notebook starts.

In [24]:
# CARRIED OVER FROM 04c SECTIONS 7.1 AND 7.5 - narrated there, not re-taught here.

CHOKE, P_ANCHOR = 30.0, 13.0
NBP_REV = 7
TIERS = (TIER_Q, TIER_M)
MIPGAP_PLAN, MIPGAP_GAME, TOL = 0.005, 1e-3, 0.5

A_INT = {(rt, p): CHOKE for rt in REGIONS for p in P}
B_SLP = {(rt, p): (CHOKE - P_ANCHOR) / DEMAND[rt, p] for rt in REGIONS for p in P}


def chain(m, r, learning, tiers, rev_price):
    """Sections 5.1-5.6 of 04c, for any region, in any model."""
    b_ = m.addVars(BUILD[r], vtype=GRB.BINARY, name=f'b_{r}')
    c_ = m.addVars(BUILD[r], lb=0.0, ub=CAP_MAX, name=f'c_{r}')
    x_ = m.addVars(ACTIVE[r], lb=0.0, name=f'x_{r}')
    fmp_ = m.addVars(P, lb=0.0, name=f'fmp_{r}')
    fpf_ = m.addVars(P, lb=0.0, name=f'fpf_{r}')
    sale_ = m.addVars(REGIONS, P, lb=0.0, name=f'sale_{r}')
    disp_ = m.addVars(P, lb=0.0, name=f'disp_{r}')

    m.addConstrs((c_[s, v] <= CAP_MAX * b_[s, v] for (s, v) in BUILD[r]), name=f'su_{r}')
    m.addConstrs((c_[s, v] >= CAP_MIN * b_[s, v] for (s, v) in BUILD[r]), name=f'sl_{r}')
    m.addConstrs((x_[s, v, p] <= (LEGACY_CAP[s, r] if v == -1 else c_[s, v])
                  for (s, v, p) in ACTIVE[r]), name=f'cap_{r}')
    m.addConstrs((gp.quicksum(ETA['MINE', v, p] * x_['MINE', v, p]
                              for v in VIN[r, 'MINE', p]) == fmp_[p] for p in P), name=f'mine_{r}')
    m.addConstrs((fmp_[p] == gp.quicksum(x_['PROC', v, p] for v in VIN[r, 'PROC', p])
                  for p in P), name=f'pin_{r}')
    m.addConstrs((gp.quicksum(ETA['PROC', v, p] * x_['PROC', v, p]
                              for v in VIN[r, 'PROC', p]) == fpf_[p] for p in P), name=f'pout_{r}')
    m.addConstrs((fpf_[p] == gp.quicksum(x_['MFG', v, p] for v in VIN[r, 'MFG', p])
                  for p in P), name=f'min_{r}')
    m.addConstrs((gp.quicksum(ETA['MFG', v, p] * x_['MFG', v, p] for v in VIN[r, 'MFG', p])
                  == sale_.sum('*', p) + disp_[p] for p in P), name=f'mout_{r}')

    cum_ = m.addVars(P, lb=0.0, ub=3 * CAP_MAX * HORIZON + EXPERIENCE0[r], name=f'cum_{r}')
    m.addConstrs((cum_[p] == EXPERIENCE0[r] +
                  gp.quicksum(LEN[q] * x_['MFG', v, q] for q in P if q <= p
                              for v in VIN[r, 'MFG', q]) for p in P), name=f'cp_{r}')

    capex_ = (gp.quicksum(MU[s, v] * FIXED[s, r] * b_[s, v] for (s, v) in BUILD[r])
              + gp.quicksum(MU[s, v] * UNIT[s, r] * c_[s, v]
                            for (s, v) in BUILD[r] if s not in LEARN_STAGES))
    if learning in ('capacity', 'both'):
        Q_ = m.addVars(P, lb=Q_START, ub=Q_START + Q_ADD, name=f'Q_{r}')
        C_ = m.addVars(P, lb=0.0, name=f'C_{r}')
        lam_ = m.addVars(P, K, lb=0.0, ub=1.0, name=f'lam_{r}')
        m.addConstrs((lam_.sum(p, '*') == 1 for p in P), name=f'sc_{r}')
        m.addConstrs((Q_[p] == gp.quicksum(QBP[k] * lam_[p, k] for k in K) for p in P), name=f'sQ_{r}')
        m.addConstrs((C_[p] == gp.quicksum(CBP[k] * lam_[p, k] for k in K) for p in P), name=f'sC_{r}')
        m.addConstrs((Q_[p] == Q_START + gp.quicksum(c_[s, v] for (s, v) in BUILD[r]
                                                     if s in LEARN_STAGES and v <= p)
                      for p in P), name=f'cc_{r}')
        for p in P:
            m.addSOS(GRB.SOS_TYPE2, [lam_[p, k] for k in K])
        rate_ = sum(UNIT[s, r] for s in LEARN_STAGES) / len(LEARN_STAGES)
        capex_ += gp.quicksum(MU['PROC', p] * rate_ * (C_[p] - (C_[p - 1] if p > 0 else 0.0))
                              for p in P)
    else:
        capex_ += gp.quicksum(MU[s, v] * UNIT[s, r] * c_[s, v]
                              for (s, v) in BUILD[r] if s in LEARN_STAGES)

    if learning in ('production', 'both') and tiers is not None:
        tq, tm = tiers
        J_ = list(range(N_TIERS))
        z_ = m.addVars(P, J_, vtype=GRB.BINARY, name=f'z_{r}')
        m.addConstrs((z_.sum(p, '*') == 1 for p in P), name=f'ot_{r}')
        lagp = {p: YEAR_TO_P[max(1, START[p] - LAG_YEARS)] for p in P}
        bigq = 3 * CAP_MAX * HORIZON + EXPERIENCE0[r]
        m.addConstrs((cum_[lagp[p]] >= tq[r][j - 1] - bigq * (1 - z_[p, j])
                      for p in P for j in J_ if j > 0), name=f'tf_{r}')
        m.addConstrs((cum_[lagp[p]] <= tq[r][j] + bigq * (1 - z_[p, j])
                      for p in P for j in J_ if j < N_TIERS - 1), name=f'tc_{r}')
        ts_ = m.addVars(STAGES, P, J_, lb=0.0, name=f'ts_{r}')
        m.addConstrs((ts_.sum(s, p, '*') == gp.quicksum(x_[s, v, p] for v in VIN[r, s, p])
                      for s in STAGES for p in P), name=f'tss_{r}')
        m.addConstrs((ts_[s, p, j] <= 3 * CAP_MAX * z_[p, j]
                      for s in STAGES for p in P for j in J_), name=f'tl_{r}')
        opex_ = gp.quicksum(OMEGA[p] * OPEX[s, r] * tm[r][j] * ts_[s, p, j]
                            for s in STAGES for p in P for j in J_)
    else:
        z_ = None
        opex_ = gp.quicksum(OMEGA[p] * OPEX[s, r] * x_[s, v, p] for (s, v, p) in ACTIVE[r])

    trans_ = gp.quicksum(OMEGA[p] * TRANSPORT[r, rt] * sale_[rt, p]
                         for rt in REGIONS for p in P)
    dcost_ = gp.quicksum(OMEGA[p] * PEN_DISPOSE * disp_[p] for p in P)
    rev_ = (gp.quicksum(OMEGA[p] * rev_price * sale_[rt, p] for rt in REGIONS for p in P)
            if rev_price is not None else None)
    return dict(b=b_, c=c_, x=x_, sale=sale_, disp=disp_, cum=cum_, z=z_,
                capex=capex_, opex=opex_, trans=trans_, dcost=dcost_, revenue=rev_,
                cost=capex_ + opex_ + trans_ + dcost_)


def best_response_pwl(r, rival, learning, tiers, nbp_rev, mipgap, env=None):
    """04c's PIECEWISE best response - the thing being validated."""
    m = gp.Model(env=env) if env is not None else gp.Model()
    m.Params.OutputFlag = 0
    m.Params.MIPGap = mipgap
    h = chain(m, r, learning, tiers, rev_price=None)
    s_ = h['sale']
    kr = list(range(nbp_rev))
    mu_ = m.addVars(REGIONS, P, kr, lb=0.0, ub=1.0, name='mu')
    revt_ = m.addVars(REGIONS, P, lb=-GRB.INFINITY, name='revt')
    for rt in REGIONS:
        for p in P:
            q_bar = rival.get((rt, p), 0.0)
            a_eff = A_INT[rt, p] - B_SLP[rt, p] * q_bar
            smax = max(1e-6, A_INT[rt, p] / B_SLP[rt, p] - q_bar)
            Sg = [smax * k / (nbp_rev - 1) for k in kr]
            Rg = [a_eff * v - B_SLP[rt, p] * v * v for v in Sg]
            m.addConstr(mu_.sum(rt, p, '*') == 1)
            m.addConstr(s_[rt, p] == gp.quicksum(Sg[k] * mu_[rt, p, k] for k in kr))
            m.addConstr(revt_[rt, p] == gp.quicksum(Rg[k] * mu_[rt, p, k] for k in kr))
    rev_ = gp.quicksum(OMEGA[p] * revt_[rt, p] for rt in REGIONS for p in P)
    m.setObjective(rev_ - h['cost'], GRB.MAXIMIZE)
    m.optimize()
    m._h, m._rev = h, rev_
    return m


print(f"carried over. horizon: {len(P)} periods, choke {CHOKE}, mesh {NBP_REV} points")

carried over. horizon: 3 periods, choke 30.0, mesh 7 points


## 8. The exact MIQP, by hand

New material starts here, and there is not much of it — which is the first
point worth making. The difference from the piecewise version above is that
revenue is written **as what it is**:

$$\text{revenue}_{rt,p} \;=\; \big(A_{rt,p} - B_{rt,p}\bar{q}_{rt,p}\big)\,s_{rt,p}
\;-\; B_{rt,p}\,s_{rt,p}^2$$

No `mu` interpolation weights. No convexity constraint. No breakpoint mesh. A
`gp.QuadExpr` and the same cost expression.

**One constraint survives from the mesh**, and it is worth keeping: `choke` caps
sales at the quantity where price reaches zero. The model would never *want* a
negative price, but the bound tightens the relaxation and costs nothing.

**And `env=ENV` matters here.** This is the only quadratic model in the series. On
a restricted licence a plain `gp.Model()` refuses a quadratic objective above
about 150 variables, which is exactly why section 1's `SMALL` exists.

> **Predict before you run.** The piecewise mesh has 7 breakpoints on a curve.
> Will the exact solution's profit be **higher or lower** than the piecewise
> one — and can you say which before running it, from the shape of the curve
> alone?

In [25]:
FIRM = 'R1'
zero_rival = {(rt, p): 0.0 for rt in REGIONS for p in P}

mx = gp.Model(env=ENV) if ENV is not None else gp.Model()
mx.Params.OutputFlag = 0
mx.Params.MIPGap = MIPGAP_PLAN
h = chain(mx, FIRM, 'none', None, rev_price=None)
s = h['sale']

# the one constraint kept from the mesh: price cannot go negative
mx.addConstrs((s[rt, p] <= max(0.0, A_INT[rt, p] / B_SLP[rt, p]
                               - zero_rival.get((rt, p), 0.0))
               for rt in REGIONS for p in P), name='choke')

# revenue, written as what it is
revenue = gp.QuadExpr()
for rt in REGIONS:
    for p in P:
        q_bar = zero_rival.get((rt, p), 0.0)
        revenue += OMEGA[p] * ((A_INT[rt, p] - B_SLP[rt, p] * q_bar) * s[rt, p]
                               - B_SLP[rt, p] * s[rt, p] * s[rt, p])

mx.setObjective(revenue - h['cost'], GRB.MAXIMIZE)
mx.update()

assert mx.NumQNZs > 0, "a QP with no quadratic terms is not the model we meant"
print(f"{mx.NumVars} variables, {mx.NumConstrs} constraints, "
      f"{mx.NumQNZs} quadratic terms")
print(f"against the free licence's ~150-variable quadratic cap: "
      f"{'fits' if mx.NumVars <= 150 else 'DOES NOT FIT'}")

mx.optimize()
assert mx.SolCount > 0, f"no solution; status {mx.Status}"
hand_built = mx.ObjVal
print(f"\nexact MIQP profit {hand_built:12.4f}   "
      f"sales {sum(s[rt, p].X for rt in REGIONS for p in P):9.4f}")

50 variables, 56 constraints, 6 quadratic terms
against the free licence's ~150-variable quadratic cap: fits

exact MIQP profit    4461.5589   sales  310.3776


## 9. Now the streamlined version

**This is where the notebook crosses from learning into convenience.**

You have written the exact revenue once. Section 10 needs it once more and
section 11 needs it inside a best-response loop — about twenty solves. Two cells:
the exact best response, and the iteration that calls it.

The iteration is 04c's, unchanged, including its **tolerance-based** convergence
test. That is still the right test here: the strategy is a continuous quantity
schedule, so exact matching would read MIP-gap wobble as a cycle.

In [26]:
def best_response_miqp(r, rival, learning, tiers, mipgap, env=None):
    """Section 8, for any firm against any rival schedule."""
    m = gp.Model(env=env) if env is not None else gp.Model()
    m.Params.OutputFlag = 0
    m.Params.MIPGap = mipgap
    h_ = chain(m, r, learning, tiers, rev_price=None)
    s_ = h_['sale']
    m.addConstrs((s_[rt, p] <= max(0.0, A_INT[rt, p] / B_SLP[rt, p]
                                   - rival.get((rt, p), 0.0))
                  for rt in REGIONS for p in P), name='choke')
    rev_ = gp.QuadExpr()
    for rt in REGIONS:
        for p in P:
            q_bar = rival.get((rt, p), 0.0)
            rev_ += OMEGA[p] * ((A_INT[rt, p] - B_SLP[rt, p] * q_bar) * s_[rt, p]
                                - B_SLP[rt, p] * s_[rt, p] * s_[rt, p])
    m.setObjective(rev_ - h_['cost'], GRB.MAXIMIZE)
    m.optimize()
    m._h, m._rev = h_, rev_
    return m


def iterate(kind, learning, tiers, nbp_rev, first, max_iter, tol, mipgap, env=None):
    """Iterated best response, with either formulation. Tolerance-based, as 04c."""
    def dist(a, bb):
        return max(abs(a[r][k] - bb[r][k]) for r in REGIONS for k in a[r])

    sales = {r: {(rt, p): 0.0 for rt in REGIONS for p in P} for r in REGIONS}
    hist, log = [], []
    order = [first] + [r for r in REGIONS if r != first]
    for it in range(max_iter):
        prev = {r: dict(sales[r]) for r in REGIONS}
        for r in order:
            rival = {}
            for other in REGIONS:
                if other == r:
                    continue
                for k, v in sales[other].items():
                    rival[k] = rival.get(k, 0.0) + v
            b = (best_response_miqp(r, rival, learning, tiers, mipgap, env)
                 if kind == 'exact' else
                 best_response_pwl(r, rival, learning, tiers, nbp_rev, mipgap, env))
            if b.SolCount == 0:
                return dict(status='INFEASIBLE', iters=it, log=log)
            sales[r] = {(rt, p): b._h['sale'][rt, p].X for rt in REGIONS for p in P}
            log.append(dict(iter=it, firm=r, profit=b.ObjVal,
                            sales=sum(sales[r].values())))
        cur = {r: dict(sales[r]) for r in REGIONS}
        if it > 0 and dist(cur, prev) < tol:
            return dict(status='CONVERGED', iters=it + 1, log=log, sales=sales)
        for k, past in enumerate(hist):
            if dist(cur, past) < tol:
                return dict(status='CYCLE', iters=it + 1, log=log, sales=sales)
        hist.append(cur)
    return dict(status='MAX_ITER', iters=max_iter, log=log, sales=sales)


print("best_response_miqp() and iterate() defined")

best_response_miqp() and iterate() defined


### 9.1 Does the wrapper reproduce the hand-built MIQP?

In [27]:
check = best_response_miqp(FIRM, zero_rival, 'none', None, MIPGAP_PLAN, ENV)
rel = abs(check.ObjVal - hand_built) / abs(hand_built)
print(f"hand-built (section 8): {hand_built:,.9f}")
print(f"wrapper    (section 9): {check.ObjVal:,.9f}")
assert rel < 1e-9, f"the wrapper is not the model you read; relative gap {rel:.2e}"
print(f"\nagree to {rel:.1e} - the wrap is earned")

hand-built (section 8): 4,461.558900598
wrapper    (section 9): 4,461.558900598

agree to 0.0e+00 - the wrap is earned


## 10. Validating the approximation

Run both formulations on the **same** instance and compare across mesh
densities.

Theory says the piecewise version should **understate** profit: revenue is
concave, chords lie below the curve, and we are maximising. So the approximation
is *conservative* — never optimistic. The cell asserts that rather than inviting
you to read the sign off a column.

In [28]:
MESHES = [3, 5, 7, 11, 21, 41]

rows = []
for n in MESHES:
    pw = best_response_pwl(FIRM, zero_rival, 'none', None, n, MIPGAP_PLAN, ENV)
    assert pw.SolCount > 0, f"mesh {n} found no solution"
    rows.append(dict(breakpoints=n, pwl_profit=round(pw.ObjVal, 3),
                     error=round(pw.ObjVal - hand_built, 3),
                     error_pct=round(100 * (pw.ObjVal - hand_built) / hand_built, 4),
                     sales=round(sum(pw._h['sale'][rt, p].X
                                     for rt in REGIONS for p in P), 3)))
mesh = pd.DataFrame(rows)

# the theory, as code: a chord below a concave curve can only understate a maximum
assert (mesh.error <= 1e-6).all(), \
    "a piecewise mesh OVERSTATED profit, which the concavity argument forbids"
print(f"every error is negative, as concavity requires: "
      f"{mesh.error_pct.min():.2f}% to {mesh.error_pct.max():.4f}%")
mesh

every error is negative, as concavity requires: -42.92% to -0.0769%


,breakpoints,pwl_profit,error,error_pct,sales
0,3,2546.685,-1914.874,-42.9194,346.196
1,5,4199.076,-262.483,-5.8832,236.556
2,7,4450.229,-11.330,-0.2539,315.407
3,11,4418.680,-42.878,-0.9611,283.867
4,21,4458.129,-3.430,-0.0769,310.622
5,41,4458.129,-3.430,-0.0769,310.622


**Every error is negative**, confirming the theory: the piecewise revenue never
overstates. The approximation is a valid lower bound on achievable profit, and
the assertion above would fail if it ever were not.

The magnitude falls sharply with mesh density — about −42.9% at 3 breakpoints,
−0.25% at 7, and −0.08% at 21 and beyond. But **the convergence is not
monotone**: 7 breakpoints (−0.25%) beats 11 (−0.96%). What matters is not the
number of breakpoints but whether one happens to land near the optimal quantity.
That is the same lesson as the SOS2 re-meshing in Part 3 — **placement beats
density**.

Seven breakpoints, the Part 4c default, costs about a quarter of a percent on a
single best response. That is defensible for the qualitative conclusions drawn
there.

## 11. But the error behaves differently inside a game

Everything above measured the error in **one** optimisation. Now measure it in an
**equilibrium**: run the whole best-response loop under each formulation and
compare where they land.

> **Predict before you run.** The single-solve error is negative and under a
> quarter of a percent at this mesh. What do you expect the equilibrium error to
> be — the same sign, the same magnitude, or something else?

In [29]:
MAX_ITER = 12

exact = iterate('exact', 'none', None, NBP_REV, first='R1', max_iter=MAX_ITER,
                tol=TOL, mipgap=MIPGAP_GAME, env=ENV)
pwl = iterate('pwl', 'none', None, NBP_REV, first='R1', max_iter=MAX_ITER,
              tol=TOL, mipgap=MIPGAP_GAME, env=ENV)
le = {g['firm']: g for g in exact['log'][-len(REGIONS):]}
lp = {g['firm']: g for g in pwl['log'][-len(REGIONS):]}

game = pd.DataFrame([
    dict(method='exact MIQP', status=exact['status'],
         **{f'profit_{r}': round(le[r]['profit'], 3) for r in REGIONS},
         **{f'sales_{r}': round(le[r]['sales'], 3) for r in REGIONS}),
    dict(method='piecewise linear', status=pwl['status'],
         **{f'profit_{r}': round(lp[r]['profit'], 3) for r in REGIONS},
         **{f'sales_{r}': round(lp[r]['sales'], 3) for r in REGIONS}),
])
for r in REGIONS:
    err = 100 * (lp[r]['profit'] / le[r]['profit'] - 1)
    print(f"{r}: piecewise reports {err:+6.2f}% against exact "
          f"({lp[r]['profit']:,.1f} vs {le[r]['profit']:,.1f})")

# the finding: in a game the sign FLIPS relative to the single-solve error
assert lp['R1']['profit'] > le['R1']['profit'], \
    "the in-game piecewise error did not come out positive; section 11's claim fails"
game

R1: piecewise reports  +1.96% against exact (2,307.3 vs 2,262.9)
R2: piecewise reports +12.15% against exact (1,775.2 vs 1,582.9)


,method,status,profit_R1,profit_R2,sales_R1,sales_R2
0,exact MIQP,CONVERGED,2262.921,1582.874,219.420,182.450
1,piecewise linear,CONVERGED,2307.270,1775.192,205.759,183.869


**This is the finding worth taking away.** In a single optimisation the piecewise
error is signed, bounded and small. In an **equilibrium** it is none of those.

Both methods converge, but to *different* equilibria, and the piecewise version
reports profits **above** the exact ones — the opposite sign to the single-solve
error, and by a much larger margin for R2 (+12.15%) than R1 (+1.96%).

The mechanism: each firm's approximated best response is slightly off, which
perturbs the *rival's* problem, which perturbs the response to that, and so on
around the loop. **The fixed point of a sequence of slightly-wrong maps is not
close to the fixed point of the correct maps** in any way the single-solve error
bound controls. Approximation error propagates through the equilibrium
computation rather than staying local.

The practical consequence is worth stating as a rule. A discretisation accuracy
that is perfectly adequate for one optimisation can be inadequate for a game
built out of the same optimisation. **If you must approximate inside a
best-response loop, validate at the equilibrium level, not the subproblem
level** — and if the qualitative conclusions flip between meshes, they are not
conclusions.

## 12. The full-scale game, exactly

With `SMALL = False` and a licence configured, the same loop runs on the complete
13-period model with both learning channels and no revenue approximation at all —
541 variables with 26 quadratic terms, against the restricted licence's ~150 cap.

**On the default settings the next cell prints an explanation instead of
running.** The figures quoted below come from a run with `SMALL = False` on an
academic licence (2026-09-03, 26.9 s).

In [30]:
if SMALL:
    print("Section 12 needs the full 13-period model, which does not fit a "
          "restricted\nlicence as a MIQP: 541 variables against a ~150 cap. Set "
          "SMALL = False\nwith a licence configured and re-run. The figures in "
          "the prose below come\nfrom exactly such a run.")
else:
    rows = []
    for first in REGIONS:
        r2 = iterate('exact', 'both', TIERS, NBP_REV, first=first, max_iter=16,
                     tol=TOL, mipgap=MIPGAP_GAME, env=ENV)
        last = {g['firm']: g for g in r2['log'][-len(REGIONS):]}
        rows.append(dict(first_mover=first, status=r2['status'],
                         iterations=r2['iters'],
                         **{f'profit_{r}': round(last[r]['profit'], 1)
                            for r in REGIONS},
                         **{f'sales_{r}': round(last[r]['sales'], 1)
                            for r in REGIONS}))
    display(pd.DataFrame(rows))

Section 12 needs the full 13-period model, which does not fit a restricted
licence as a MIQP: 541 variables against a ~150 cap. Set SMALL = False
with a licence configured and re-run. The figures in the prose below come
from exactly such a run.


### What the full-scale exact run said

| | piecewise (Part 4c) | exact MIQP | |
|---|---|---|---|
| total quantity | 2,276.5 | 2,264.5 | −0.5% |
| average price | 15.81 | 15.84 | +0.03 |
| joint profit | 17,791.5 | 18,461.1 | +3.8% |
| output uplift from production learning | +15.2% | **+13.4%** | both positive |
| disposal | 0 | **0** | unchanged |
| convergence | CONVERGED | **CYCLE** | ← see below |

**Every qualitative conclusion survives.** Output rises with the production
channel, price falls, the incumbent gains more than the entrant (R1 1,082.0 →
1,262.6 against R2's 914.9 → 1,001.9), and disposal stays at exactly zero. A
conclusion that survives both formulations is a conclusion about the economics.
One that does not was an artefact of the mesh.

**But the exact game does not settle.** Both move orders end in `CYCLE` — at 5
iterations from R1 and 6 from R2 — where the piecewise version converged cleanly.
The cycling profiles are close (R1 earns 11,419.6 and 11,419.9 across the two
orders, within 0.003%), so this is not a large oscillation; it is the quantity
profile refusing to sit still to within the `tol = 0.5` test.

That points the same way as section 11. The piecewise mesh **discretises the
strategy space**: quantities can only land on convex combinations of seven
breakpoints, which damps the best-response map and helps it reach a fixed point.
Remove the mesh and the map is free to keep moving. **The smoothness that made
Part 4c converge was partly an artefact of the approximation**, not a property of
the game.

So the honest reading of Part 4c's "pure-strategy equilibrium, from both move
orders" is narrower than it looks: an equilibrium *of the approximated game*. The
economics is robust; the convergence is not.

## 13. The agreement assertion

Section 8 was built by hand, and `src/lithium/` holds the same model as a
function. **The same model exists twice, deliberately** — and deliberate
duplication with nothing comparing the copies is how a bug gets fixed in three
places out of four.

In [31]:
from lithium import Instance, best_response_miqp as pkg_miqp, build_structure

nb_instance = Instance(
    regions=tuple(REGIONS), stages=tuple(STAGES),
    fixed=FIXED, unit=UNIT, opex=OPEX,
    legacy_cap=LEGACY_CAP, legacy_ret=LEGACY_RET,
    eta_ceil=ETA_CEIL, eta_base=ETA_BASE, alpha=ALPHA, beta=BETA, delta_bar=DELTA_BAR,
    demand_base=DEMAND_BASE, demand_growth=DEMAND_GROWTH, experience0=EXPERIENCE0,
)
nb_struct = build_structure(nb_instance, blocks=BLOCKS, dr=DR, life=LIFE, lead=LEAD,
                            cap_min=CAP_MIN, cap_max=CAP_MAX,
                            legacy_byr=LEGACY_BYR, eta_floor=ETA_FLOOR)

packaged = pkg_miqp(
    FIRM, zero_rival, nb_struct,
    a_int=A_INT, b_slp=B_SLP, learning='none', mipgap=MIPGAP_PLAN, env=ENV,
    transport=TRANSPORT, pen_dispose=PEN_DISPOSE, price_fixed=PRICE_FIXED,
    capex_curve=(QBP, CBP), learn_stages=LEARN_STAGES,
    n_tiers=N_TIERS, lag_years=LAG_YEARS,
)

rel = abs(packaged.ObjVal - hand_built) / abs(hand_built)
print(f"notebook (section 8, by hand): {hand_built:,.9f}")
print(f"package  (lithium.games)     : {packaged.ObjVal:,.9f}")
assert rel < 1e-9, f"notebook and package disagree by {rel:.2e}"
print(f"\nnotebook and package agree to {rel:.1e}")
print(f"(both at {len(P)} periods, because BLOCKS follows SMALL - so this check")
print(" runs at whatever scale you set, and needs no licence at the default.)")

notebook (section 8, by hand): 4,461.558900598
package  (lithium.games)     : 4,461.558900598

notebook and package agree to 0.0e+00
(both at 3 periods, because BLOCKS follows SMALL - so this check
 runs at whatever scale you set, and needs no licence at the default.)


## 14. Summary

| Question | Answer |
|---|---|
| Does the piecewise mesh understate profit? | **Yes, always** — concave revenue, maximised, so chords lie below |
| How much, at 7 breakpoints? | About **−0.25%** on a single best response |
| Does the error shrink monotonically with density? | **No** — 7 beats 11. Placement beats density |
| Does the same bound hold in a game? | **No.** The sign flips: piecewise reports **+1.96%** for R1 and **+12.15%** for R2 |
| Do Part 4c's conclusions survive exact solution? | The **economics** does. The **convergence** does not — the exact game cycles |
| Does any of this need a licence? | Only section 12. At `SMALL = True` the MIQP is 50 variables |

### Formulation lessons

- **An error bound proved for one optimisation says nothing about a fixed point
  built from it.** Validate at the equilibrium level.
- **Placement beats density.** A mesh point near the optimum is worth more than
  ten spread evenly.
- **Approximation can manufacture convergence.** Discretising the strategy space
  damps the best-response map; the exact game cycles where the approximated one
  settles.
- **Check the model is the kind you meant.** `assert mx.NumQNZs > 0` in section 8
  catches a "QP" that is secretly linear — which would validate nothing.

### Things to try

- `MESHES = [6, 7, 8, 9, 10]` — zoom in on the non-monotonicity and see how much
  of it is luck
- `SMALL = False` with a licence — reproduce section 12 rather than reading it
- `NBP_REV = 3` in section 11 — a coarse mesh inside the loop; does the *sign* of
  the in-game error survive, or does the equilibrium move somewhere else entirely?
- `learning='both'` in section 10 — validate the approximation with the tier
  binaries present, which is what Part 4c actually runs